In [1]:
%matplotlib qt
import matplotlib.pyplot as plt
import mne
import numpy as np
import os
import pandas as pd
from scipy.io import loadmat

In [2]:
def import_brainstorm_digitizer_data_to_mne(fpath_dig):
    """
    Reads Brainstorm Digitizer data (mat-file) to generate a MNE montage object.
    """
    # Load digitized data
    data = loadmat(fpath_dig)

    # Extract channel, headshape and fiducial locations
    #    Channel locations
    df_channel_locs = pd.DataFrame(data["Channel"].transpose().flatten())
    ch_pos = {u[0]: v.flatten() for u, v in zip(df_channel_locs["Name"], df_channel_locs["Loc"])}

    #    Headspace and fiducial locations
    df_headpoint_locs = pd.DataFrame(data["HeadPoints"].flatten())
    lbls_hp = [w[0][0] for w in df_headpoint_locs["Label"][0].transpose()]
    locs_hp = [w for w in df_headpoint_locs["Loc"][0].transpose()]
    nasion = np.average([loc for loc, lbl in zip(locs_hp, lbls_hp) if lbl == "NA"], axis=0)
    lpa = np.average([loc for loc, lbl in zip(locs_hp, lbls_hp) if lbl == "LPA"], axis=0)
    rpa = np.average([loc for loc, lbl in zip(locs_hp, lbls_hp) if lbl == "RPA"], axis=0)
    hsp = np.array([loc for loc, lbl in zip(locs_hp, lbls_hp) if lbl == "EXTRA"])

    # Create montage
    dig = mne.channels.make_dig_montage(ch_pos=ch_pos, nasion=nasion, lpa=lpa, rpa=rpa, hsp=hsp)

    return dig

# Given

In [3]:
data_dir = os.path.expanduser('~/data/EEGAcamp/')
sfreq = 1000
head_radius = .58 / (2 * np.pi) # head circumference / 2π
# fname_raw = "sub-01_imotions.csv"
fname_dig = "sub-01.mat"
# fname_raw = "001_Anonymous 02-10-24 14h18m.csv"
# fname_raw = "001_Anonymous 18-09-24 13h52m.csv"
# fname_raw = "001_Amira Ahamed.csv"
fname_raw = "001_Amira_09-10-24.csv"

In [4]:
# Import raw data in CSV file and convert to a formatted dataframe
#   Read data from Excel
fpath_raw = os.path.join(data_dir, "iMotions", fname_raw)
df_raw = pd.read_csv(fpath_raw, low_memory=False)

# Drop general info at top
inx_first_timepoint = df_raw.index[df_raw.iloc[:, 0] == '1'].to_list()[0]
inx_header = inx_first_timepoint - 1
df_raw.drop(df_raw.head(inx_header).index, inplace=True)

#   Change header
header = df_raw.iloc[0].values
df_raw = df_raw[1:]
df_raw.columns = header
df_raw

,Row,Timestamp,EventSource,SlideEvent,StimType,Duration,CollectionPhase,SourceStimuliName,EventSource,SampleNumber,...,ET_CameraLeftY,ET_CameraRightX,ET_CameraRightY,ET_ValidityLeft,ET_ValidityRight,EventSource,Event Group,Event Label,Event Text,Event Index
30,1,345.6441,SlideEvents,StartSlide,TestImage,3000000,StimuliDisplay,Screen recording,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31,2,346.53195,NaN,NaN,NaN,NaN,NaN,Screen recording,ActiCHamp,462493,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32,3,346.6418,NaN,NaN,NaN,NaN,NaN,Screen recording,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,4,347.53195,NaN,NaN,NaN,NaN,NaN,Screen recording,ActiCHamp,462494,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,5,347.6393,NaN,NaN,NaN,NaN,NaN,Screen recording,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423400,423371,318458.53195,NaN,NaN,NaN,NaN,NaN,Screen recording,ActiCHamp,780605,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
423401,423372,318458.8974,NaN,NaN,NaN,NaN,NaN,Screen recording,NaN,NaN,...,-1,-1,-1,4,4,NaN,NaN,NaN,NaN,NaN
423402,423373,318459.53195,NaN,NaN,NaN,NaN,NaN,Screen recording,ActiCHamp,780606,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
423403,423374,318459.5571,SlideEvents,EndMedia,TestImage,3000000,StimuliDisplay,Screen recording,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df_raw["Blink"].dtypes

dtype('O')

In [9]:
np.unique(df_raw["Blink"].astype(float))

array([ 0.,  1., nan])

In [10]:
plt.figure()
plt.plot(df_raw["Blink"].astype(float), '.')
plt.show()
# df_raw["Blink"]

In [ ]:
# #   Extract LSL triggers (note: Timestamp are in msecs and LSL Timestamp are in secs)
# df_triggers = df_raw.drop(df_raw.index[df_raw['LSL Timestamp'].isna()])
# df_triggers.dropna(axis=1, how='all', inplace=True)  # drop columns that don't have any data left
# df_triggers.drop(columns=['Row', 'Combined Event Source', 'SourceStimuliName'], inplace=True)
# df_triggers['Timestamp'] = df_triggers['Timestamp'].astype('float')
# df_triggers.reset_index(drop=True, inplace=True)
# df_triggers

In [11]:
#   Extract EEG data
df_eeg = df_raw.drop(df_raw.index[df_raw['Fp1'].isna()])    # drop rows without EEG data
df_eeg.dropna(axis=1, how='all', inplace=True)  # drop columns that don't have any data left
df_eeg.drop(columns=['Row', 'SourceStimuliName', 'SampleNumber'],
            inplace=True)
aux_ch_names = ['Aux%d' % w for w in range(1, 9)] + ['Channel %d' % w for w in range(33, 41)]
for col_name in ['Combined Event Source', 'EventSource'] + aux_ch_names:
    try:
        df_eeg.drop(columns=[col_name], inplace=True)
    except:
        pass
df_eeg = df_eeg.astype('float')
df_eeg.reset_index(drop=True, inplace=True)
df_eeg

,Timestamp,Fp1,Fz,F3,F7,T7,C3,Cz,Ref,P7,...,Extra3,Extra4,Extra5,Extra6,Extra7,Extra8,Extra9,Extra10,Extra11,Extra12
0,346.53195,-3420.214757,-1843.701125,9131.103285,-2669.775323,-1510.839806,12648.925462,7064.990056,4189.794816,4621.728399,...,39376.317365,34885.741306,30830.272659,35878.954172,32338.133949,34046.678827,38162.352552,32185.448406,32793.065578,27546.141882
1,347.53195,-3382.421790,-1807.373001,9178.759534,-2629.833918,-1469.335900,12692.724289,7106.005680,4229.443253,4666.699101,...,39416.747051,34927.342868,30871.727736,35921.239327,32382.030432,34087.889764,38202.049816,32233.934733,32843.944483,27566.063757
2,348.53195,-3344.970619,-1766.406205,9227.880626,-2585.400325,-1423.583948,12738.134444,7147.216616,4272.119033,4714.550662,...,39461.620097,34970.213960,30914.647657,35966.454170,32428.221837,34128.758903,38245.409190,32301.512856,32892.919091,27606.298131
3,349.53195,-3299.658120,-1728.710894,9281.298594,-2538.867123,-1376.269496,12783.446943,7190.234193,4315.429578,4761.669802,...,39504.881815,35012.108491,30956.737499,36009.520575,32475.633945,34167.528434,38286.131845,32347.557777,32938.964012,27659.276645
4,350.53195,-3261.962808,-1688.085895,9320.702890,-2497.607359,-1332.275357,12830.126629,7232.128724,4358.007702,4805.273316,...,39547.069313,35051.366302,30996.678904,36054.491277,32517.967929,34201.171011,38328.026376,32376.756995,32983.104636,27703.856722
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
318109,318455.53195,-3101.025312,-1150.488252,9441.747808,-2080.810494,-939.355445,13465.868800,7656.738088,5281.591663,5092.040887,...,40500.877883,35554.588946,31392.382019,36727.879932,32220.311686,33696.532352,38865.575190,32714.647611,33576.952277,26214.696603
318110,318456.53195,-3095.312422,-1165.136689,9435.351324,-2089.990182,-952.978492,13447.460598,7640.038869,5266.503773,5073.339716,...,40482.957962,35535.155352,31373.778504,36708.202198,32203.173015,33682.323368,38848.729487,32699.022611,33558.690558,26214.354807
318111,318457.53195,-3114.013593,-1184.326142,9426.757574,-2112.841744,-977.441382,13422.753567,7618.749808,5250.439320,5048.681513,...,40455.907181,35509.276447,31348.632021,36683.641651,32177.391765,33658.788212,38827.147457,32676.317534,33536.913215,26202.684885
318112,318458.53195,-3154.150311,-1213.330047,9418.847418,-2145.556586,-1011.425756,13398.730130,7592.187308,5226.025259,5018.701045,...,40424.413041,35479.491291,31317.479678,36654.002980,32147.997235,33629.051885,38796.190426,32643.456207,33507.616341,26169.970042


In [ ]:
# Convert dataframe to numpy array in SI units
eeg_data = df_eeg.iloc[:, 1:].values.T / 1e6    # in Volts

# Create MNE info object for the data
ch_names = df_eeg.columns.tolist()[1:]
ch_types = ['eeg'] * len(ch_names)
info = mne.create_info(ch_names, sfreq, ch_types=ch_types)

# Create MNE raw object for the data
raw = mne.io.RawArray(eeg_data, info)
# raw.crop(0, 12)

# Add GND channel
mne.add_reference_channels(raw, ref_channels=['GND'], copy=False)
display(raw)

# Import BS digitizer data to MNE montage
fpath_dig = os.path.join(data_dir, "Digitization", fname_dig)
dig = import_brainstorm_digitizer_data_to_mne(fpath_dig)

# Combine MNE montage with raw data
raw.set_montage(dig)

# Visualize digitized data
dig.plot(sphere=head_radius)       # 2d
fig = dig.plot(kind="3d")   # 3d
mne.viz.plot_alignment(
    raw.info,
    dig=True,
    eeg=True,
)   # 3D plot

# Drop ground channel (in future, we might use this; for now, we drop it after merging dig and raw)
raw.drop_channels('GND')

# Plot time series and frequency power spectra
raw.plot(scalings='auto', duration=5)
raw.compute_psd(fmax=100).plot(sphere=head_radius)
raw.compute_psd().plot_topo()

# Filter raw data and plot time series and power spectra
raw_filtered = raw.copy().filter(l_freq=None, h_freq=50)
raw_filtered.plot(scalings='auto', duration=5)
raw_filtered.compute_psd(fmax=100).plot(sphere=head_radius)
raw_filtered.compute_psd().plot_topo()

# Export raw data in FIF format
fpath_exportedraw = os.path.join(data_dir, "iMotions", "sub-01_raw_eeg.fif")
raw.save(fpath_exportedraw, overwrite=True)